# 時間複雜度實測手冊

這份 Notebook 搭配「時間與空間複雜度入門」筆記，提供 `timeit` 量測模板，幫助你比較不同演算法的成長趨勢。


## 使用方式
1. 決定要比較的函式（例如：線性搜尋 vs. 內建查找）。
2. 以 `sizes` 指定輸入規模，觀察耗時如何隨著 `n` 變化。
3. 將結果整理成表格或圖表（若想繪圖，可自行匯入 `matplotlib`）。

本 Notebook 預設提供三個示例函式：
- `sum_linear`：單迴圈，理論 `O(n)`。
- `contains_linear`：線性搜尋，理論 `O(n)`。
- `contains_set`：使用 `set`，理論期望 `O(1)` 查找。
- `nested_loop`：雙層迴圈，理論 `O(n^2)`。


In [5]:
import random
import timeit
from typing import Callable, Dict, Iterable, List

ResultRow = Dict[str, float]

random.seed(42)

def build_list(n: int) -> List[int]:
    return list(range(n))

def sum_linear(data: List[int]) -> int:
    total = 0
    for value in data:
        total += value
    return total

def contains_linear(data: List[int], target: int) -> bool:
    for value in data:
        if value == target:
            return True
    return False

def contains_set(data: List[int], target: int) -> bool:
    data_set = set(data)
    return target in data_set

def nested_loop(data: List[int]) -> int:
    count = 0
    for i in range(len(data)):
        for j in range(len(data)):
            if data[i] <= data[j]:
                count += 1
    return count

def bench(func: Callable, setup_args: Dict) -> List[ResultRow]:
    sizes = setup_args["sizes"]
    trials = setup_args.get("trials", 3)
    results: List[ResultRow] = []
    for n in sizes:
        data = build_list(n)
        stmt = setup_args["stmt_builder"](func, data)
        timer = timeit.Timer(stmt, globals={**globals(), "data": data, "func": func, "n": n})
        elapsed = min(timer.repeat(repeat=trials, number=setup_args.get("number", 1)))
        results.append({"n": n, "seconds": elapsed})
    return results

def print_results(title: str, rows: List[ResultRow]) -> None:
    print(title)
    for row in rows:
        print(f"n={row['n']:>6} → {row['seconds']:.6f} sec")
    print()


## 範例：比較線性搜尋與集合查找
執行下列程式，觀察在不同 `n` 下的耗時差異。預期集合查找維持近似常數時間，而線性搜尋會隨 `n` 成長。


In [6]:
linear_results = bench(
    contains_linear,
    {
        "sizes": [1_000, 5_000, 10_000, 20_000],
        "stmt_builder": lambda func, data: "func(data, n - 1)",
        "trials": 3,
    },
)
print_results("線性搜尋", linear_results)

set_results = bench(
    contains_set,
    {
        "sizes": [1_000, 5_000, 10_000, 20_000],
        "stmt_builder": lambda func, data: "func(data, n - 1)",
        "trials": 3,
    },
)
print_results("集合查找", set_results)


線性搜尋
n=  1000 → 0.000006 sec
n=  5000 → 0.000027 sec
n= 10000 → 0.000059 sec
n= 20000 → 0.000124 sec

集合查找
n=  1000 → 0.000013 sec
n=  5000 → 0.000056 sec
n= 10000 → 0.000102 sec
n= 20000 → 0.000236 sec



## 範例：觀察 `O(n)` 與 `O(n^2)` 差別
以下比較 `sum_linear` 與 `nested_loop`。雖然兩者都很簡單，但雙層迴圈在輸入變大時會迅速變慢。


In [7]:
linear_sum_results = bench(
    sum_linear,
    {
        "sizes": [500, 1_000, 5_000, 10_000],
        "stmt_builder": lambda func, data: "func(data)",
        "trials": 3,
    },
)
print_results("線性累加", linear_sum_results)

nested_results = bench(
    nested_loop,
    {
        "sizes": [100, 200, 400],
        "stmt_builder": lambda func, data: "func(data)",
        "trials": 3,
    },
)
print_results("雙層迴圈", nested_results)


線性累加
n=   500 → 0.000006 sec
n=  1000 → 0.000011 sec
n=  5000 → 0.000053 sec
n= 10000 → 0.000116 sec

雙層迴圈
n=   100 → 0.000167 sec
n=   200 → 0.000669 sec
n=   400 → 0.002839 sec



## 自己動手
- 將 `sizes` 改成更大的數字，體驗耗時差異。
- 將 `stmt_builder` 改成其它測試語句，例如：排序、字典查找、字串拼接。
- 若要視覺化，可收集 `results` 後用 `matplotlib` 繪圖。
- 思考結果是否符合你對複雜度的預期；若不符合，檢查是否存在額外成本（例如：`set` 建立是 `O(n)`）。
